In [0]:
%pip install phonenumbers pycountry rapidfuzz

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql import Window


In [0]:
#Lecture Bronze
df = spark.table("workspace.final_project.bronze_orders")
display(df)

In [0]:
# Identify and handle full duplicated lines
# Define a window partitioned by all columns to detect duplicates
window_all = Window.partitionBy(df.columns).orderBy(F.lit(1))

# Add a row number within each duplicate group
df_with_rownum = df.withColumn("row_num", F.row_number().over(window_all))

# Separate first occurrences (keep) and full duplicates (quarantine)
df_first_occurrences = df_with_rownum.filter(F.col("row_num") == 1).drop("row_num")
df_full_duplicates = df_with_rownum.filter(F.col("row_num") > 1).drop("row_num")

# Add quarantine_reason and timestamp for fully duplicated lines
df_full_duplicates = df_full_duplicates.withColumn("quarantine_reason", F.lit("full_duplicated_line")) \
                                       .withColumn("quarantine_timestamp", F.current_timestamp())

# Initialize df_orders_quarantine with full duplicates
df_orders_quarantine = df_full_duplicates

# Define df_valid with cleaned order_id and supplier_id
# Keep only rows where order_id and supplier_id are not null and order_id is not an invalid string
df_valid = df_first_occurrences.filter(
    (F.col("order_id").isNotNull()) &         # real null check
    (F.trim(F.col("order_id")) != "") &       # remove empty strings
    (F.lower(F.col("order_id")) != "null") &  # remove string "null"
    (F.lower(F.col("order_id")) != "none") &  # remove string "none"
    (F.col("supplier_id").isNotNull())        # supplier_id must not be null
)

# Add remaining invalid rows (not in df_valid) to quarantine
df_remaining_quarantine = df_first_occurrences.join(
    df_valid.select("order_id", "supplier_id"), 
    on=["order_id", "supplier_id"], 
    how="left_anti"  # keep only rows not in df_valid
)

# Add quarantine_reason and timestamp for these remaining invalid rows
df_remaining_quarantine = df_remaining_quarantine.withColumn(
    "quarantine_reason",
    F.when(F.col("order_id").isNull() & F.col("supplier_id").isNull(), "missing_order_id_and_supplier_id")
     .when(F.col("order_id").isNull(), "missing_order_id")
     .when(F.col("supplier_id").isNull(), "missing_supplier_id")
     .otherwise("invalid_order_id_or_supplier_id")  # other invalid cases
).withColumn(
    "quarantine_timestamp",
    F.current_timestamp()
)

# Combine full duplicates and other quarantined rows
df_orders_quarantine = df_orders_quarantine.unionByName(df_remaining_quarantine)

In [0]:
display(df_valid)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# 1️⃣ Nettoyer les colonnes : NULL si vide
df_clean = df.withColumn(
    "order_id_clean",
    F.when(F.col("order_id").isNull() | (F.col("order_id") == ""), None)
     .otherwise(F.col("order_id"))
).withColumn(
    "supplier_id_clean",
    F.when(F.col("supplier_id").isNull() | (F.col("supplier_id") == ""), None)
     .otherwise(F.col("supplier_id"))
)

# 2️⃣ Supprimer .0 final
df_clean = df_clean.withColumn(
    "order_id_clean",
    F.regexp_replace(F.col("order_id_clean"), r"\.0$", "")
).withColumn(
    "supplier_id_clean",
    F.regexp_replace(F.col("supplier_id_clean"), r"\.0$", "")
)

# 3️⃣ Créer une UDF pour convertir en int ou retourner None
def safe_int(x):
    try:
        return int(x)
    except:
        return None

from pyspark.sql.types import IntegerType
from pyspark.sql.functions import udf

safe_int_udf = udf(safe_int, IntegerType())

df_clean = df_clean.withColumn("order_id_int", safe_int_udf(F.col("order_id_clean"))) \
                   .withColumn("supplier_id_int", safe_int_udf(F.col("supplier_id_clean")))

# 4️⃣ Séparer valides / invalides
df_valid = df_clean.filter(F.col("order_id_int").isNotNull() & F.col("supplier_id_int").isNotNull())

df_orders_quarantine = df_clean.filter(F.col("order_id_int").isNull() | F.col("supplier_id_int").isNull()) \
    .withColumn(
        "quarantine_reason",
        F.when(F.col("order_id_int").isNull() & F.col("supplier_id_int").isNull(),
               "invalid_order_id_and_supplier_id")
         .when(F.col("order_id_int").isNull(), "invalid_order_id")
         .when(F.col("supplier_id_int").isNull(), "invalid_supplier_id")
    ).withColumn(
        "quarantine_timestamp", F.current_timestamp()
    )

In [0]:
# Split of items and orders dataset
# Select columns for orders
df_orders = df_valid.select(
    "order_id",
    "supplier_id",
    "order_date",
    "delivery_date_expected",
    "delivery_date_actual"
)
# Select columns for items
df_items = df_valid.select(
    "order_id",
    F.posexplode("items").alias("item_index", "item")
)

# Flatten items
df_items = df_items.select(
    "order_id",
    "item_index",
    "item.product_category",
    "item.quantity",
    "item.unit_price"
)

# Add item_id
# We add a unique identifier for each item with hash 
df_items = df_items.withColumn(
    "item_id",
    F.sha2(F.concat_ws("||", F.col("order_id"), F.col("item_index")), 256)
)

In [0]:
from pyspark.sql import functions as F

# List of known formats in your data
date_formats = [
    "yyyy-MM-dd",           # iso
    "yyyy-M-d",             # iso_no_zero
    "yyyy/MM/dd",           # slash
    "dd-MM-yyyy",           # dash_fr
    "yyyy-MM-dd'T'HH:mm:ss" # datetime
]

# Function to parse a column using multiple formats
def parse_multiformat_date(column):
    """
    Try parsing a column with multiple known formats.
    Returns a timestamp column or null if no format matches.
    """
    col_expr = None
    for fmt in date_formats:
        new_expr = F.expr(f"try_to_timestamp({column}, '{fmt}')")
        if col_expr is None:
            col_expr = new_expr
        else:
            col_expr = F.coalesce(col_expr, new_expr)
    return col_expr

# Apply parsing to all date columns
df_clean = df_orders \
    .withColumn("order_date_ts", parse_multiformat_date("order_date")) \
    .withColumn("delivery_date_expected_ts", parse_multiformat_date("delivery_date_expected")) \
    .withColumn("delivery_date_actual_ts", parse_multiformat_date("delivery_date_actual"))

# Optional: check which rows could not be parsed
df_clean.select(
    "order_date", "order_date_ts",
    "delivery_date_expected", "delivery_date_expected_ts",
    "delivery_date_actual", "delivery_date_actual_ts"
).show(20, truncate=False)


In [0]:
df_orders.show(15)

In [0]:
#Profiling rapide
null_counts = df.select([
    F.count(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
            c
        )
    ).alias(c)
    for c in df.columns
])

display(null_counts)

#doublons exacts
total_rows = df.count()
distinct_rows = df.distinct().count()

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Exact duplicates:", total_rows - distinct_rows)

#Doublons exacts
duplicate_rows = df.groupBy(df.columns[1:]).count().filter(F.col("count") > 1).drop("count")
display(duplicate_rows)
